In [ ]:
# | default_exp cli

In [ ]:
# | export
import typer
from pathlib import Path
from typing import Optional
import pandas as pd
import structlog

log = structlog.get_logger()
app = typer.Typer(name="kreview", help="ctDNA fragmentomics feature evaluation")


def version_callback(value: bool):
    if value:
        import kreview

        typer.echo(f"kreview version: {kreview.__version__}")
        raise typer.Exit()


@app.callback()
def main(
    version: Optional[bool] = typer.Option(
        None, "--version", callback=version_callback, is_eager=True
    ),
):
    """Kreview: ctDNA fragmentomics feature evaluation framework."""
    pass


In [ ]:
# | export
@app.command()
def label(
    cancer_samplesheet: Path = typer.Option(..., help="Cancer samplesheet CSV"),
    healthy_xs1_samplesheet: Path = typer.Option(
        ..., help="Healthy XS1 samplesheet CSV"
    ),
    healthy_xs2_samplesheet: Path = typer.Option(
        ..., help="Healthy XS2 samplesheet CSV"
    ),
    cbioportal_dir: Path = typer.Option(..., help="Directory with cBioPortal files"),
    output: Path = typer.Option("labels.parquet", help="Output parquet file"),
    min_vaf: float = typer.Option(
        0.01, help="Min VAF for Possible ctDNA+ (default 1%)"
    ),
    min_variants: int = typer.Option(
        1, help="Min # variants passing VAF for Possible ctDNA+"
    ),
    chunk_size: int = typer.Option(
        500,
        "--chunk-size",
        help="Batch size for DuckDB file loading over SFTP network mounts",
    ),
):
    """Generate ctDNA labels without feature evaluation."""
    from kreview.core import Paths, LabelConfig
    from kreview.labels import CtDNALabeler

    print("=== kreview label ===", flush=True)
    print("Configuration:", flush=True)
    print(f"  --cancer-samplesheet : {cancer_samplesheet}", flush=True)
    print(f"  --healthy-xs1       : {healthy_xs1_samplesheet}", flush=True)
    print(f"  --healthy-xs2       : {healthy_xs2_samplesheet}", flush=True)
    print(f"  --cbioportal-dir    : {cbioportal_dir}", flush=True)
    print(f"  --output            : {output}", flush=True)
    print(f"  --min-vaf           : {min_vaf}", flush=True)
    print(f"  --min-variants      : {min_variants}", flush=True)
    print(f"  --chunk-size        : {chunk_size}", flush=True)
    print("", flush=True)

    paths = Paths(
        str(cancer_samplesheet),
        str(healthy_xs1_samplesheet),
        str(healthy_xs2_samplesheet),
        str(cbioportal_dir),
        [],
    )
    config = LabelConfig(
        min_vaf=min_vaf, min_variants=min_variants, chunk_size=chunk_size
    )

    labeler = CtDNALabeler(paths, config)
    labels = labeler.label_all()
    labels.to_parquet(output, index=False)
    typer.echo(f"Labels written to {output}")
    typer.echo(labels["label"].value_counts().to_string())


In [ ]:
# | export
@app.command()
def features_list():
    """List all registered feature evaluators."""
    from kreview.registry import get_all_evaluators

    evals = get_all_evaluators()
    for e in evals:
        typer.echo(
            f"Tier {getattr(e, 'tier', '?')}: {getattr(e, 'name', type(e).__name__)} ({getattr(e, 'source_file', '?')})"
        )


In [ ]:
# | export
import numpy as np
import sys
import time
from concurrent.futures import ProcessPoolExecutor
import concurrent.futures


def _impute(df, strategy: str):
    """Apply imputation strategy to a feature DataFrame.

    Args:
        df: DataFrame with numeric features (may contain NaN).
        strategy: One of 'zero', 'mean', 'median'.

    Returns:
        DataFrame with NaN values filled according to strategy.
    """
    if strategy == "mean":
        return df.fillna(df.mean())
    elif strategy == "median":
        return df.fillna(df.median())
    elif strategy != "zero":
        import structlog

        structlog.get_logger().warning(
            "unknown_impute_strategy",
            strategy=strategy,
            fallback="zero",
        )
    return df.fillna(0)  # default: zero


def _extract_from_dataframe(e_name, df_chunk, verbose=False):
    """Extract features from a pre-loaded DataFrame chunk. Runs in child process.

    Key design: The parent process loads the full cohort from DuckDB ONCE,
    then shards the DataFrame to workers. This avoids N parallel glob scans
    over network mounts which causes I/O deadlock.
    """
    from kreview.registry import get_all_evaluators
    import sys

    def _log(msg, **kw):
        extras = " ".join(f"{k}={v}" for k, v in kw.items())
        ts = time.strftime("%H:%M:%S")
        print(f"[{ts}] [worker] {msg}  {extras}", file=sys.stderr, flush=True)

    evaluators = {ev.name: ev for ev in get_all_evaluators()}
    if e_name not in evaluators:
        _log("ERROR: evaluator_not_found", name=e_name)
        return e_name, False, f"Evaluator '{e_name}' not found in registry"
    e = evaluators[e_name]

    sid_col = "sample_id" if "sample_id" in df_chunk.columns else "SAMPLE_ID"
    uniq_sids = df_chunk[sid_col].unique()
    _log("chunk_started", evaluator=e_name, samples=len(uniq_sids), rows=len(df_chunk))

    try:
        t0 = time.time()
        extracted_rows = []
        for i, sample_id in enumerate(uniq_sids):
            samp_data = df_chunk[df_chunk[sid_col] == sample_id]
            res = e.extract(samp_data)
            if res is not None:
                if isinstance(res, pd.DataFrame):
                    res["SAMPLE_ID"] = sample_id
                    extracted_rows.append(res)
                else:
                    res["SAMPLE_ID"] = sample_id
                    extracted_rows.append(pd.DataFrame([res]))

            if verbose and i > 0 and i % 200 == 0:
                _log("extracting", evaluator=e_name, processed=i, total=len(uniq_sids))

        feat_matrix = (
            pd.concat(extracted_rows, ignore_index=True)
            if extracted_rows
            else pd.DataFrame()
        )
        total_sec = time.time() - t0
        _log(
            "chunk_completed",
            evaluator=e_name,
            matrix_rows=len(feat_matrix),
            cols=len(feat_matrix.columns) if not feat_matrix.empty else 0,
            seconds=f"{total_sec:.1f}",
        )
        return e_name, True, feat_matrix

    except Exception as exc:
        import traceback

        _log("CHUNK_CRASHED", evaluator=e_name, error=str(exc))
        traceback.print_exc(file=sys.stderr)
        sys.stderr.flush()
        return e_name, False, str(exc)


def _render_quarto_report(
    matrix_path: str,
    feat_name: str,
    report_dir: Path,
    python_exe: str,
    cvd_safe: bool = False,
    shap_samples: int = 500,
    shap_features: int = 10,
) -> tuple[bool, str]:
    """Render a Quarto HTML dashboard for a single feature matrix."""
    import subprocess
    import os

    pkg_dir = Path(__file__).resolve().parent
    template_dir = pkg_dir / "templates"
    template_file = template_dir / "report_template.qmd"

    if not template_file.exists():
        return False, f"Template not found at {template_file}"

    env = os.environ.copy()
    env["QUARTO_PYTHON"] = str(python_exe)

    out_html = Path(report_dir) / f"{feat_name}_dashboard.html"
    cmd = [
        "quarto",
        "render",
        "report_template.qmd",
        "-P",
        f"matrix_path:{Path(matrix_path).absolute()}",
        "-P",
        f"feature_name:{feat_name}",
        "-P",
        f"cvd_safe:{str(cvd_safe).lower()}",
        "-P",
        f"shap_samples:{shap_samples}",
        "-P",
        f"shap_features:{shap_features}",
        "--output",
        out_html.name,
        "--output-dir",
        str(out_html.parent.absolute()),
    ]

    try:
        subprocess.run(
            cmd,
            env=env,
            capture_output=True,
            text=True,
            check=True,
            timeout=600,
            cwd=str(template_dir),
        )
        return True, str(out_html)
    except subprocess.CalledProcessError as exc:
        return False, exc.stderr[-1500:]
    except subprocess.TimeoutExpired:
        return False, "Timeout (>600s)"


@app.command()
def run(
    cancer_samplesheet: Path = typer.Option(...),
    healthy_xs1_samplesheet: Path = typer.Option(...),
    healthy_xs2_samplesheet: Path = typer.Option(...),
    cbioportal_dir: Path = typer.Option(...),
    krewlyzer_dir: list[str] = typer.Option(..., help="krewlyzer output directory"),
    output: Path = typer.Option("output/", help="Output directory"),
    min_vaf: float = typer.Option(0.01),
    min_fragments: int = typer.Option(2000),
    min_variants: int = typer.Option(1),
    features: Optional[str] = typer.Option(
        None, help="Comma-separated features to run"
    ),
    tier: Optional[int] = typer.Option(None, help="Run features of this tier only"),
    workers: int = typer.Option(4, help="Total processes"),
    verbose: bool = typer.Option(
        False, "--verbose", "-v", help="Enable verbose logging"
    ),
    cvd_safe: bool = typer.Option(
        False,
        "--cvd-safe",
        help="Render dashboards and plots using an Okabe-Ito Colorblind-Safe palette instead of default neon.",
    ),
    skip_report: bool = typer.Option(False, help="Skip HTML report generation"),
    cv_folds: int = typer.Option(
        5, "--cv-folds", help="Number of cross-validation folds (3-20, default 5)"
    ),
    impute_strategy: str = typer.Option(
        "median",
        "--impute-strategy",
        help="Imputation strategy for missing values: median, mean, or zero",
    ),
    export_duckdb: bool = typer.Option(
        False,
        "--export-duckdb",
        help="Export a persistent duckdb data lake containing all feature matrices",
    ),
    chunk_size: int = typer.Option(
        500,
        "--chunk-size",
        help="Batch size for DuckDB file loading over SFTP network mounts",
    ),
    top_n: int = typer.Option(
        50,
        "--top-n",
        help="Max sub-metrics to feed into ML models per evaluator. All sub-metrics are included up to this cap; the model's feature importance ranks them.",
    ),
    shap_samples: int = typer.Option(
        500,
        "--shap-samples",
        help="Max samples for SHAP explainability computation in dashboards. Lower values reduce memory usage.",
    ),
    shap_features: int = typer.Option(
        10,
        "--shap-features",
        help="Max features to visualize in SHAP beeswarm/waterfall plots. The model still trains on --top-n features.",
    ),
    resume: bool = typer.Option(
        False,
        "--resume",
        help="Skip evaluators whose model results already exist in the output directory.",
    ),
    compute_univariate_auc: bool = typer.Option(
        False,
        "--compute-univariate-auc",
        help="Compute per-feature univariate LR AUC (adds ~10s per evaluator).",
    ),
):
    """Run full pipeline: label → extract → evaluate → report."""
    from kreview.core import Paths, LabelConfig, load_feature_cohort
    from kreview.labels import CtDNALabeler
    from kreview.registry import get_all_evaluators

    from kreview.eval_engine import evaluate_feature, single_feature_model
    import time
    import glob
    import json

    def _echo(msg):
        print(msg, flush=True)

    if cv_folds < 3 or cv_folds > 20:
        _echo("ERROR: --cv-folds must be between 3 and 20")
        raise typer.Exit(code=1)
    if impute_strategy not in ["median", "mean", "zero"]:
        _echo("ERROR: --impute-strategy must be median, mean, or zero")
        raise typer.Exit(code=1)

    _echo(f"=== kreview run (verbose={verbose}, workers={workers}) ===")
    _echo("Configuration:")
    _echo(f"  --cancer-samplesheet : {cancer_samplesheet}")
    _echo(f"  --healthy-xs1       : {healthy_xs1_samplesheet}")
    _echo(f"  --healthy-xs2       : {healthy_xs2_samplesheet}")
    _echo(f"  --cbioportal-dir    : {cbioportal_dir}")
    _echo(f"  --krewlyzer-dir     : {krewlyzer_dir}")
    _echo(f"  --output            : {output}")
    _echo(f"  --min-vaf           : {min_vaf}")
    _echo(f"  --min-variants      : {min_variants}")
    _echo(f"  --features          : {features or 'ALL'}")
    _echo(f"  --tier              : {tier or 'ALL'}")
    _echo(f"  --workers           : {workers}")
    _echo(f"  --cv-folds          : {cv_folds}")
    _echo(f"  --top-n             : {top_n}")
    _echo(f"  --impute-strategy   : {impute_strategy}")
    _echo(f"  --chunk-size        : {chunk_size}")
    _echo(f"  --shap-samples      : {shap_samples}")
    _echo(f"  --shap-features     : {shap_features}")
    _echo(f"  --skip-report       : {skip_report}")
    _echo(f"  --cvd-safe          : {cvd_safe}")
    _echo(f"  --export-duckdb     : {export_duckdb}")
    _echo(f"  --resume            : {resume}")
    _echo(f"  --compute-univariate-auc : {compute_univariate_auc}")
    _echo(f"  --verbose           : {verbose}")
    _echo("")

    # ── Step 1: Labels ──
    _echo("Step 1: Generating Labels...")
    t0 = time.time()
    paths = Paths(
        str(cancer_samplesheet),
        str(healthy_xs1_samplesheet),
        str(healthy_xs2_samplesheet),
        str(cbioportal_dir),
        list(krewlyzer_dir),
    )
    config = LabelConfig(
        min_vaf=min_vaf, min_variants=min_variants, chunk_size=chunk_size
    )
    labeler = CtDNALabeler(paths, config)
    labels_df = labeler.label_all()
    _echo(f"  Labels: {len(labels_df)} samples in {time.time()-t0:.1f}s")
    _echo(f"  Distribution:\n{labels_df['label'].value_counts().to_string()}")

    # ── Step 2: Resolve evaluators ──
    _echo("Step 2: Resolving Feature Evaluators...")
    all_evals = get_all_evaluators()
    target_evals = []
    feat_filter = features.split(",") if features else []
    for e in all_evals:
        if tier is not None and getattr(e, "tier", -1) != tier:
            continue
        if feat_filter and getattr(e, "name", "") not in feat_filter:
            continue
        target_evals.append(e)

    if not target_evals:
        _echo(f"ERROR: No evaluators matched. Available: {[e.name for e in all_evals]}")
        raise typer.Exit(code=1)
    _echo(f"  Matched: {[e.name for e in target_evals]}")

    out_path = Path(output)
    out_path.mkdir(parents=True, exist_ok=True)
    all_sample_ids = list(labels_df["SAMPLE_ID"].unique())

    for e in target_evals:
        _echo(f"\n{'='*60}")
        _echo(f"Processing evaluator: {e.name}")

        # ── Resume checkpoint: skip if model results already exist ──
        if resume:
            checkpoint = out_path / f"{e.name}_model_results.json"
            if checkpoint.exists():
                _echo(f"  SKIP (--resume): {checkpoint.name} already exists")
                continue

        # ── Step 3: Load + Shard + Extract ──
        _echo(f"Step 3: Loading & extracting '{e.name}'...")
        t1 = time.time()
        _echo(f"  Loading cohort from DuckDB (chunk_size={chunk_size})...")
        df_full = load_feature_cohort(
            str(e.source_file),
            paths.krewlyzer_dirs,
            set(all_sample_ids),
            chunk_size=chunk_size,
        )

        if df_full.empty:
            _echo(f"  WARNING: No data found for {e.name}, skipping")
            continue

        sid_col = "sample_id" if "sample_id" in df_full.columns else "SAMPLE_ID"
        n_samples = df_full[sid_col].nunique()
        _echo(
            f"  Loaded: {n_samples} samples, {len(df_full)} rows in {time.time()-t1:.1f}s"
        )

        # Shard by sample_id
        unique_ids = df_full[sid_col].unique()
        n_chunks = min(workers * 2, len(unique_ids))
        id_chunks = np.array_split(unique_ids, n_chunks)
        df_chunks = [df_full[df_full[sid_col].isin(set(chunk))] for chunk in id_chunks]
        _echo(f"  Sharded into {len(df_chunks)} chunks")

        t2 = time.time()
        with ProcessPoolExecutor(max_workers=workers) as pool:
            future_map = {}
            for ci, chunk_df in enumerate(df_chunks):
                fut = pool.submit(_extract_from_dataframe, e.name, chunk_df, verbose)
                future_map[fut] = ci

            _echo(f"  Submitted {len(future_map)} tasks to {workers} workers")

            partial_results = []
            failed_samples = []
            completed = 0
            for fut in concurrent.futures.as_completed(future_map):
                chunk_idx = future_map[fut]
                completed += 1
                try:
                    e_name, success, data = fut.result()
                    if success and not data.empty:
                        partial_results.append(data)
                        _echo(
                            f"  [{completed}/{len(future_map)}] chunk {chunk_idx}: OK ({len(data)} rows)"
                        )
                    elif success:
                        _echo(
                            f"  [{completed}/{len(future_map)}] chunk {chunk_idx}: OK (empty)"
                        )
                    else:
                        _echo(
                            f"  [{completed}/{len(future_map)}] chunk {chunk_idx}: FAILED - {data}"
                        )
                        failed_chunk = id_chunks[chunk_idx]
                        failed_samples.extend(list(failed_chunk))
                except Exception as exc:
                    _echo(
                        f"  [{completed}/{len(future_map)}] chunk {chunk_idx}: EXCEPTION - {exc}"
                    )
                    failed_chunk = id_chunks[chunk_idx]
                    failed_samples.extend(list(failed_chunk))

        extract_sec = time.time() - t2
        if failed_samples:
            failed_path = out_path / f"{e.name}_failed_samples.csv"
            pd.DataFrame({"SAMPLE_ID": failed_samples}).to_csv(failed_path, index=False)
            _echo(
                f"  WARNING: {len(failed_samples)} samples failed extraction. Saved to {failed_path.name}"
            )
        _echo(f"  Extraction complete in {extract_sec:.1f}s")

        if not partial_results:
            _echo(f"  WARNING: No results for {e.name}, skipping")
            continue

        feat_matrix = pd.concat(partial_results, ignore_index=True)
        merged = pd.merge(labels_df, feat_matrix, on="SAMPLE_ID", how="inner")
        out_p = out_path / f"{e.name}_matrix.parquet"
        merged.to_parquet(out_p, index=False)
        _echo(
            f"  Matrix: {merged.shape[0]} samples x {merged.shape[1]} cols -> {out_p}"
        )

        # ── Step 4: Statistical Evaluation ──
        _echo(f"Step 4: Running statistical evaluation for '{e.name}'...")
        t3 = time.time()

        # Identify numeric feature columns (exclude label/metadata cols)
        meta_cols = set(labels_df.columns) | {"SAMPLE_ID", "sample_id", "filename"}
        numeric_cols = [
            c
            for c in merged.select_dtypes(include=np.number).columns
            if c not in meta_cols
        ]
        _echo(f"  Found {len(numeric_cols)} numeric feature columns")

        eval_results = []
        for col in numeric_cols:
            res = evaluate_feature(
                merged[col],
                merged["label"],
                total_fragments=merged.get("n_total_somatic_snvs"),
                max_vaf=merged.get("max_vaf"),
            )
            res["feature_column"] = col
            eval_results.append(res)

        eval_df = pd.DataFrame(eval_results)

        # Opt-in: compute univariate AUC per feature column
        if compute_univariate_auc:
            from kreview.eval_engine import univariate_auc as _uauc

            _echo(f"  Computing univariate AUC for {len(numeric_cols)} features...")
            # Build binary target for univariate AUC (same as model target)
            _model_mask = merged["label"].isin(
                [
                    "True ctDNA+",
                    "Possible ctDNA+",
                    "Healthy Normal",
                    "Possible ctDNA-",
                    "Possible ctDNA\u2212",
                ]
            )
            _m_df = merged[_model_mask]
            _y_uauc = (
                _m_df["label"]
                .isin(["True ctDNA+", "Possible ctDNA+"])
                .astype(int)
                .values
            )
            uauc_scores = []
            for col in numeric_cols:
                auc_val = _uauc(_m_df[col], _y_uauc, n_folds=cv_folds)
                uauc_scores.append(auc_val)
            eval_df["univariate_auc"] = uauc_scores

        eval_out = out_path / f"{e.name}_eval_stats.parquet"
        eval_df.to_parquet(eval_out, index=False)
        _echo(f"  Eval stats: {eval_df.shape[0]} features -> {eval_out}")

        # Single-feature model (4-Tier Inclusion: Positives vs Negatives)
        model_label = merged["label"].isin(
            [
                "True ctDNA+",
                "Possible ctDNA+",
                "Healthy Normal",
                "Possible ctDNA-",
                "Possible ctDNA−",
            ]
        )
        model_df = merged[model_label].copy()

        y = (
            model_df["label"]
            .isin(["True ctDNA+", "Possible ctDNA+"])
            .astype(int)
            .values
        )
        if len(model_df) >= 20 and len(np.unique(y)) == 2:
            # Pre-filter sub-metrics by Cohen's d (noise reduction) then
            # feed the top --top-n into the model for importance ranking.
            if "cohens_d_true_vs_healthy" in eval_df.columns:
                ranked = (
                    eval_df.dropna(subset=["cohens_d_true_vs_healthy"])
                    .sort_values("cohens_d_true_vs_healthy", key=abs, ascending=False)
                    .head(top_n)["feature_column"]
                    .tolist()
                )
                top_feats = ranked if ranked else numeric_cols[:top_n]
            else:
                top_feats = numeric_cols[:top_n]
            _echo(
                f"  Feeding {len(top_feats)}/{len(numeric_cols)} sub-metrics into model (--top-n={top_n})"
            )

            if top_feats:
                # Variance guard: drop features that are constant across all model samples.
                # Constant features produce AUC=0.500 and waste compute.
                X_raw = _impute(model_df[top_feats], impute_strategy)
                nonconst = [c for c in top_feats if X_raw[c].std() > 0]
                n_dropped = len(top_feats) - len(nonconst)
                if n_dropped > 0:
                    _echo(
                        f"  WARNING: Dropped {n_dropped} constant features (zero variance)"
                    )
                    top_feats = nonconst

                if not top_feats:
                    _echo(
                        f"  WARNING: All features are constant for {e.name}, skipping model"
                    )
                    continue

                import warnings

                warnings.filterwarnings("ignore", message=".*use_label_encoder.*")

                _echo(f"  Imputation: {impute_strategy}")

                X = _impute(model_df[top_feats], impute_strategy).values
                c_types = model_df.get("CANCER_TYPE", None)
                if c_types is not None:
                    c_types = c_types.values
                a_types = model_df.get("access_version", None)
                if a_types is not None:
                    a_types = a_types.values

                model_res, lr_model, rf_model, xgb_model = single_feature_model(
                    X,
                    y,
                    feature_names=top_feats,
                    cancer_types=c_types,
                    assays=a_types,
                    n_folds=cv_folds,
                )
                model_out = out_path / f"{e.name}_model_results.json"

                if "error" not in model_res:
                    model_res["top_features"] = top_feats

                import joblib

                # Save models for downstream SHAP / Dashboards
                if lr_model is not None:
                    joblib_out = out_path / f"{e.name}_lr_model.joblib"
                    joblib.dump(lr_model, joblib_out)
                if rf_model is not None:
                    joblib_out = out_path / f"{e.name}_rf_model.joblib"
                    joblib.dump(rf_model, joblib_out)
                if xgb_model is not None:
                    joblib_out = out_path / f"{e.name}_xgb_model.joblib"
                    joblib.dump(xgb_model, joblib_out)

                with open(model_out, "w") as f:
                    json.dump(model_res, f, indent=2, default=str)

                def _fmt(v):
                    return "N/A" if v is None else f"{v:.3f}"

                _echo(
                    f"  Model: AUC_LR={_fmt(model_res.get('auc_lr'))}, "
                    f"AUC_RF={_fmt(model_res.get('auc_rf'))}, "
                    f"AUC_XGB={_fmt(model_res.get('auc_xgb'))} -> {model_out}"
                )

        eval_sec = time.time() - t3
        _echo(f"  Evaluation complete in {eval_sec:.1f}s")

    # ── Step 4b: Build Cross-Evaluator Scoreboard ──
    from kreview.scoreboard import build_scoreboard

    sb = build_scoreboard(out_path)
    if not sb.empty:
        sb.to_parquet(out_path / "scoreboard_combined__all.parquet")
        sb.to_csv(out_path / "scoreboard_combined__all.csv", index=False)
        _echo(f"\n  Scoreboard: {len(sb)} evaluators ranked")
        _echo(f"  Top 5 by AUC:")
        for _, row in sb.head(5).iterrows():
            best = row["best_auc"]
            auc_str = f"{best:.3f}" if not pd.isna(best) else "N/A"
            _echo(f"    {row['evaluator']:<30} AUC={auc_str}")
        _echo(f"  -> scoreboard_combined__all.parquet")

    # ── Step 5: Generate HTML Dashboards ──
    if not skip_report:
        report_dir = out_path / "reports"
        report_dir.mkdir(parents=True, exist_ok=True)

        matrices = glob.glob(str(out_path / "*_matrix.parquet"))
        if matrices:
            _echo(f"\nStep 5: Generating {len(matrices)} HTML Dashboard(s)...")
            for p in matrices:
                feat_name = Path(p).name.replace("_matrix.parquet", "")
                _echo(f"  Rendering {feat_name}...")
                ok, msg = _render_quarto_report(
                    p,
                    feat_name,
                    report_dir,
                    sys.executable,
                    cvd_safe=cvd_safe,
                    shap_samples=shap_samples,
                    shap_features=shap_features,
                )
                if ok:
                    _echo(f"  Saved: {msg}")
                else:
                    _echo(f"  Quarto FAILED for {feat_name}: {msg}")

    # ── Step 6: DuckLake Export ──
    if export_duckdb:
        _echo("\nStep 6: Exporting Unified DuckLake...")
        import duckdb

        db_path = out_path / "kreview_lake.duckdb"
        if db_path.exists():
            db_path.unlink()
        try:
            con = duckdb.connect(str(db_path))
            matrices = glob.glob(str(out_path / "*_matrix.parquet"))
            for p in matrices:
                table_name = (
                    Path(p).name.replace("_matrix.parquet", "").replace(".", "_")
                )
                _echo(f"  Importing {table_name} into DuckLake...")
                con.execute(
                    f"CREATE TABLE {table_name} AS SELECT * FROM read_parquet('{p}')"
                )
            con.close()
            _echo(f"  DuckLake saved securely directly to: {db_path}")
        except Exception as e:
            _echo(f"  DuckLake creation failed: {e}")

    total_sec = time.time() - t0
    _echo(f"\n=== Workflow complete in {total_sec:.1f}s ===")


In [ ]:
# | export
@app.command()
def report(
    input_dir: Path = typer.Option(..., help="Directory with *_matrix.parquet files"),
    out_dir: Path = typer.Option(
        "reports/", help="Directory to deposit Quarto reports"
    ),
    cvd_safe: bool = typer.Option(
        False,
        "--cvd-safe",
        help="Render dashboards and plots using an Okabe-Ito Colorblind-Safe palette instead of default neon.",
    ),
    shap_samples: int = typer.Option(
        500,
        "--shap-samples",
        help="Max samples for SHAP explainability computation in dashboards.",
    ),
    shap_features: int = typer.Option(
        10,
        "--shap-features",
        help="Max features to visualize in SHAP plots.",
    ),
):
    """Re-generate HTML Dashboards from existing matrix parquet files."""
    import glob
    import sys

    print("=== kreview report ===", flush=True)
    print("Configuration:", flush=True)
    print(f"  --input-dir     : {input_dir}", flush=True)
    print(f"  --out-dir       : {out_dir}", flush=True)
    print(f"  --cvd-safe      : {cvd_safe}", flush=True)
    print(f"  --shap-samples  : {shap_samples}", flush=True)
    print(f"  --shap-features : {shap_features}", flush=True)
    print("", flush=True)

    in_path = Path(input_dir).absolute()
    matrices = glob.glob(str(in_path / "*_matrix.parquet"))
    if not matrices:
        print(f"No *_matrix.parquet files found in {in_path}", flush=True)
        return

    out_path = Path(out_dir).absolute()
    out_path.mkdir(parents=True, exist_ok=True)

    for p in matrices:
        feat_name = Path(p).name.replace("_matrix.parquet", "")
        print(f"Rendering dashboard for {feat_name}...", flush=True)
        ok, msg = _render_quarto_report(
            p,
            feat_name,
            out_path,
            sys.executable,
            cvd_safe=cvd_safe,
            shap_samples=shap_samples,
            shap_features=shap_features,
        )
        if ok:
            print(f"  Saved: {msg}", flush=True)
        else:
            print(f"  FAILED: {msg}", flush=True)

In [ ]:
# | test
# Smoke test
assert hasattr(app, "registered_commands")